# Contextual Multi-Armed Bandit

For the contextual multi-armed bandit (cMAB) when user information is available (context), we implemented a generalisation of Thompson sampling algorithm ([Agrawal and Goyal, 2014](https://arxiv.org/pdf/1209.3352.pdf)) based on NumPyro.

![title](img/cmab.png)

The following notebook contains an example of usage of the class Cmab, which implements the algorithm above.

In [1]:
import numpy as np

from pybandits.cmab import CmabBernoulli
from pybandits.model import BayesianNeuralNetwork, BnnLayerParams, BnnParams, FeaturesConfig, StudentTArray

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
n_samples = 1000
n_features = 5

First, we need to define the input context matrix $X$ of size ($n\_samples, n\_features$) and the mapping of possible actions $a_i \in A$ to their associated model.

In [3]:
# context
X = 2 * np.random.random_sample((n_samples, n_features)) - 1  # random float in the interval (-1, 1)
print("X: context matrix of shape (n_samples, n_features)")
print(X[:10])

X: context matrix of shape (n_samples, n_features)
[[-0.78084264 -0.3991762  -0.98650122  0.86674257  0.39524681]
 [-0.79453883  0.35573449 -0.72330194  0.15255431 -0.47885515]
 [-0.54047846  0.59545972 -0.00497222  0.48052656 -0.81008558]
 [ 0.92760116 -0.41458209 -0.81306075  0.01558444  0.80235384]
 [ 0.57717885  0.31335463 -0.76287426 -0.24320256 -0.84563826]
 [ 0.35803771 -0.48910245 -0.91193463 -0.01514885  0.66717584]
 [ 0.05369483 -0.55325707  0.21784675  0.3744297  -0.6333678 ]
 [-0.90878112  0.1988578  -0.27895583 -0.20461197 -0.94630956]
 [-0.1346903   0.5917763  -0.74111887  0.33326867  0.92977281]
 [-0.56336649  0.32641353 -0.05142436  0.80441486  0.05621856]]


In [4]:
# define action model
bias = StudentTArray.cold_start(mu=1, sigma=2, shape=1)
weight = StudentTArray.cold_start(shape=(n_features, 1))
layer_params = BnnLayerParams(weight=weight, bias=bias)
model_params = BnnParams(bnn_layer_params=[layer_params])
feature_config = FeaturesConfig(n_features=n_features)

update_kwargs = {"num_steps": 100, "batch_size": 128, "optimizer_type": "adam"}

actions = {
    "a1": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_kwargs=update_kwargs,
    ),
    "a2": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_kwargs=update_kwargs,
    ),
}

We can now init the bandit given the mapping of actions $a_i$ to their model.

In [5]:
# init contextual Multi-Armed Bandit model
cmab = CmabBernoulli(actions=actions)

The predict function below returns the action selected by the bandit at time $t$: $a_t = argmax_k P(r=1|\beta_k, x_t)$. The bandit selects one action per each sample of the contect matrix $X$.

In [6]:
# predict action
pred_actions, _, _ = cmab.predict(X)
print("Recommended action: {}".format(pred_actions[:10]))

Recommended action: ['a1', 'a1', 'a1', 'a2', 'a2', 'a2', 'a1', 'a2', 'a1', 'a2']


Now, we observe the rewards and the context from the environment. In this example rewards and the context are randomly simulated.

In [7]:
# simulate reward from environment
simulated_rewards = np.random.randint(2, size=n_samples).tolist()
print("Simulated rewards: {}".format(simulated_rewards[:10]))

Simulated rewards: [1, 1, 0, 1, 0, 0, 0, 1, 0, 1]


Finally, we update the model providing per each action sample: (i) its context $x_t$ (ii) the action $a_t$ selected by the bandit, (iii) the corresponding reward $r_t$.

In [8]:
# update model
cmab.update(context=X, actions=pred_actions, rewards=simulated_rewards)

SVI:   0%|          | 0/34 [00:00<?, ?it/s]

SVI:   3%|▎         | 1/34 [00:01<00:35,  1.07s/it]

SVI:   3%|▎         | 1/34 [00:01<00:35,  1.07s/it, loss=3195.1465]

SVI:   6%|▌         | 2/34 [00:01<00:34,  1.07s/it, loss=2206.1133]

SVI:   9%|▉         | 3/34 [00:01<00:33,  1.07s/it, loss=3004.4883]

...

SVI:  91%|█████████ | 31/34 [00:00<00:02,  1.06it/s, loss=2655.8489]

SVI:  94%|█████████▍| 32/34 [00:00<00:01,  1.06it/s, loss=2923.2668]

SVI:  97%|█████████▋| 33/34 [00:01<00:00,  1.06it/s, loss=2075.7891]

SVI: 100%|██████████| 34/34 [00:01<00:00, 22.02it/s, loss=2075.7891]

SVI: 100%|██████████| 34/34 [00:01<00:00, 22.02it/s, loss=2497.0625]